# CarRacingWObstacles: PPO in Google Colab

This notebook installs the environment, trains PPO for **500 completed training episodes**,
plots total reward per episode (raw and trailing mean over 100 episodes), and displays
an MP4 rollout of the **best validation checkpoint**. Run the cells in order.

In Colab choose **Runtime → Change runtime type → T4 GPU** if available. The baseline uses one RGB frame and updates the action every physics frame.

Validation runs every 25 training episodes on a frozen **5+5 scenario suite**.
These additional validation episodes do not count toward the 500 training episodes.
The ten final test cases are used only after checkpoint selection. A successful policy
is an experimental outcome; 500 episodes is the training budget, not a success guarantee.

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"] = "1"

REPO = Path("/content/CarRacingWObstacles")
if not (REPO / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "https://github.com/luisgarrote/CarRacingWObstacles.git", str(REPO)])
else:
    # Update an existing clean checkout without overwriting student changes.
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only"])
# SWIG builds Box2D; setuptools <74 retains a module used by legacy NumPy/Cython.
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade",
                       "pip", "setuptools>=68,<74", "wheel", "swig"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-build-isolation",
                       str(REPO) + "[sb3]"])
# A normal installation is importable in this already-running Colab kernel.
importlib.invalidate_caches()


In [ ]:
import csv, json, math, time, sys, subprocess
import importlib.metadata
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Video, Image as DisplayImage, Markdown
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

# Make the cloned simulator importable without restarting the kernel.
sys.path.insert(0, str(REPO))
EXAMPLES = REPO / "examples"
sys.path.insert(0, str(EXAMPLES))
from env_utils import episode, get_case, load_cases, load_validation_cases, save_results
from colab_callbacks import ValidateAndStop

from action_repeat import ActionRepeat
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecTransposeImage
from car_racing_obstacles import CarRacingObstacles

torch.set_num_threads(1)


## Experiment settings

Change the seed and output directory for each independent run. Keep validation
and final test seeds out of training. This demo uses one environment so the
episode callback counts exactly 500 episodes across the run.

`HISTORY = 1` and `ACTION_REPEAT = 1` are the baseline defaults. Investigate and
justify these parameters as part of your experiments; their best values are not supplied.


In [ ]:
TRAIN_EPISODES = 500
SEED = 0
HISTORY = 1
ACTION_REPEAT = 1
MAX_STEPS = 1000  # physics frames per episode
ROLLOUT = 512
BATCH_SIZE = 128
EPOCHS = 4
VALIDATE_EVERY = 25
VALIDATION_CASES = load_validation_cases()
CONTINUOUS = True
RANDOMIZE = False
TRAFFIC = 0  # set 1 or 2 to train with moving obstacles
OBSTACLES = 0
POLICY = "CnnPolicy"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN = REPO / "outputs" / ("colab_ppo_seed" + str(SEED) + "_" + str(time.time_ns()))
RUN.mkdir(parents=True)
print("Device:", DEVICE, "| Output:", RUN)

## Validation scenarios: five random + five random appearance

- **normal:** five random tracks with fixed appearance (seeds 10001–10005).
- **randomized:** five random tracks with appearance randomization (seeds 11001–11005).

These ten cases are frozen in `examples/validation_cases.json` so every checkpoint
sees the same cases. They have no added road hazards. The two groups use different
tracks; appearance randomization also affects the RNG sequence used for track generation.

Best-model selection uses the **mean raw return over all ten cases**, so each
five-case group contributes equally. The notebook also reports each group's
return, standard deviation and success rate. The final test suite uses different
seeds (20001–20005 and 21001–21005) and is excluded from checkpoint selection.


In [ ]:
def show_table(columns, rows):
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    lines += ["| " + " | ".join(str(value) for value in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

validation_seeds = {case["seed"] for case in VALIDATION_CASES}
assert SEED not in validation_seeds
assert not validation_seeds.intersection(case["seed"] for case in load_cases())
case_rows = [
    [case["id"], case["group"], case["seed"],
     "randomized" if case["env_kwargs"]["domain_randomize"] else "fixed"]
    for case in VALIDATION_CASES
]
show_table(["Case", "Group", "Seed", "Appearance"], case_rows)

## Construct the environment and show every wrapper

`CarRacingObstacles` is instantiated directly below. `ENV_KWARGS` is the plain
parameter dictionary passed to the constructor. A zero-argument factory is used
because SB3's `DummyVecEnv` expects one. The action-repeat, history and image-axis
wrappers are written out in order. Keep the environment and reward model fixed
for the main algorithm comparison; record any separate changes.


In [ ]:
# All simulator parameters used by this notebook are visible here.
ENV_KWARGS = {
    "continuous": CONTINUOUS, "domain_randomize": RANDOMIZE,
    "n_static_obstacles": OBSTACLES, "n_dynamic_obstacles": TRAFFIC,
    "n_mountains": 0, "use_ghost": False,
    "lap_complete_percent": 0.95, "obstacle_scale": 0.1,
    "render_mode": "rgb_array",
}

def make_training_env():
    env = CarRacingObstacles(max_episode_steps=MAX_STEPS, **ENV_KWARGS)
    env = ActionRepeat(env, ACTION_REPEAT)
    return Monitor(env, filename=str(RUN / "training"))

train_env = DummyVecEnv([make_training_env])
if HISTORY > 1:
    train_env = VecFrameStack(train_env, n_stack=HISTORY, channels_order="last")
train_env = VecTransposeImage(train_env)  # RGB HWC -> CNN CHW; SB3 scales uint8 pixels.
train_env.seed(SEED)
print("Observation:", train_env.observation_space)
print("Action:", train_env.action_space)

# Each validation case overrides only its scene/appearance settings.
def make_validation_env(case):
    case_kwargs = dict(ENV_KWARGS)
    case_kwargs.update(case["env_kwargs"])
    case_kwargs["continuous"] = CONTINUOUS
    env = CarRacingObstacles(max_episode_steps=case["max_steps"], **case_kwargs)
    env = ActionRepeat(env, ACTION_REPEAT)
    vec = DummyVecEnv([lambda: env])
    if HISTORY > 1:
        vec = VecFrameStack(vec, n_stack=HISTORY, channels_order="last")
    return VecTransposeImage(vec)

## Save the experiment settings

The JSON records the environment settings, PPO parameters, package versions and source revision.

In [ ]:
# Reproducibility metadata is separate from the learning code.
config = {
    "algorithm": "PPO", "policy": POLICY, "seed": SEED,
    "history": HISTORY, "action_repeat": ACTION_REPEAT, "episodes": TRAIN_EPISODES,
    "max_steps": MAX_STEPS, "device": DEVICE, "environment": ENV_KWARGS,
    "randomize": RANDOMIZE, "traffic": TRAFFIC, "obstacles": OBSTACLES,
    "validate_every": VALIDATE_EVERY,
    "validation_case_ids": [case["id"] for case in VALIDATION_CASES],
}
config["ppo"] = {"n_steps": ROLLOUT, "batch_size": BATCH_SIZE, "n_epochs": EPOCHS, "learning_rate": 3e-4}
config["versions"] = {name: importlib.metadata.version(name) for name in ("stable-baselines3", "torch", "gymnasium", "numpy")}
config["repository_commit"] = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
(RUN / "config.json").write_text(json.dumps(config, indent=2))


## Train PPO and select the best checkpoint

The PPO calls below are ordinary SB3. `ValidateAndStop` evaluates deterministic
actions on all ten validation cases, saves `best_model.zip` whenever mean raw
validation return across the 5+5 suite improves, and stops at 500 completed training episodes.
See `examples/colab_callbacks.py` for its short implementation.

`model.zip` stores the final policy; the video below explicitly loads
`best_model.zip`. Training and validation rewards are never normalized.

In [ ]:
callback = ValidateAndStop(RUN, VALIDATION_CASES, make_validation_env, max_episodes=TRAIN_EPISODES, every=VALIDATE_EVERY)
# SB3 counts policy decisions; this upper bound lets the callback stop at 500 episodes.
decision_budget = TRAIN_EPISODES * math.ceil(MAX_STEPS / ACTION_REPEAT) + ROLLOUT
try:
    model = PPO(POLICY, train_env, n_steps=ROLLOUT, batch_size=BATCH_SIZE, n_epochs=EPOCHS,
                learning_rate=3e-4, seed=SEED, device=DEVICE, verbose=0)
    model.learn(total_timesteps=decision_budget, callback=callback)
    model.save(str(RUN / "model"))
    config["completed_episodes"] = len(callback.training_returns)
    config["actual_timesteps"] = model.num_timesteps
    config["simulator_frames"] = callback.simulator_frames
    (RUN / "config.json").write_text(json.dumps(config, indent=2))
finally:
    train_env.close()
assert config["completed_episodes"] == TRAIN_EPISODES
print("Best checkpoint:", json.loads((RUN / "best_model.json").read_text()))

## Total reward per completed training episode

The thin curve shows each episode's raw reward sum. The thick curve is the mean
of the most recent 100 episodes; before episode 100 it uses the available episodes.
The validation episodes are not included in this training graph.

In [ ]:
with (RUN / "training.monitor.csv").open() as stream:
    rows = list(csv.DictReader(line for line in stream if not line.startswith("#")))
rewards = np.array([float(row["r"]) for row in rows])
episodes = np.arange(1, len(rewards) + 1)
average = np.array([rewards[max(0, i - 99):i + 1].mean() for i in range(len(rewards))])
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(episodes, rewards, alpha=0.4, linewidth=1, label="Raw episode return")
ax.plot(episodes, average, linewidth=2, label="Mean of last 100 episodes")
ax.set(xlabel="Completed training episode", ylabel="Total reward")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
fig.savefig(RUN / "rewards.png", dpi=160)
display(DisplayImage(filename=str(RUN / "rewards.png")))
plt.close(fig)
reward_data = np.column_stack([episodes, rewards, average])
np.savetxt(RUN / "rewards.csv", reward_data, delimiter=",", header="episode,raw_return,mean_last_100", comments="")

## Validation results and best checkpoint

The tables show the selected checkpoint's five + five validation results. The
curves compare the two groups and their combined mean across checkpoints.
Individual failed episodes are retained in `validation.csv`; group summaries
are saved in `validation_summary.csv`.


In [ ]:
best_info = json.loads((RUN / "best_model.json").read_text())
print(f"Best after {best_info['training_episodes']} episodes; mean validation return: {best_info['mean_return']:.2f}")
group_rows = [
    [group, stats["episodes"], f"{stats['mean_return']:.2f}", f"{stats['std_return']:.2f}",
     f"{stats['success_rate']:.0%}"] for group, stats in best_info["groups"].items()
]
show_table(["Group", "Cases", "Mean return", "Std. return", "Success rate"], group_rows)
with (RUN / "validation.csv").open() as stream:
    validation_rows = list(csv.DictReader(stream))
selected_rows = [row for row in validation_rows
                 if int(row["training_episodes"]) == best_info["training_episodes"]]
selected_result_rows = [
    [row["case_id"], row["group"], row["seed"], f"{float(row['episode_return']):.2f}",
     row["length"], row["success"]] for row in selected_rows
]
show_table(["Case", "Group", "Seed", "Return", "Decisions", "Success"], selected_result_rows)
with (RUN / "validation_summary.csv").open() as stream:
    validation_curve = list(csv.DictReader(stream))
fig, ax = plt.subplots(figsize=(10, 4))
groups = ["overall"] + sorted({row["group"] for row in validation_curve} - {"overall"})
for group in groups:
    rows = [row for row in validation_curve if row["group"] == group]
    x = [int(row["training_episodes"]) for row in rows]
    y = [float(row["mean_return"]) for row in rows]
    ax.plot(x, y, marker="o", label=group)
ax.set(xlabel="Completed training episode", ylabel="Mean raw validation return")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
fig.savefig(RUN / "validation_rewards.png", dpi=160)
display(DisplayImage(filename=str(RUN / "validation_rewards.png")))
plt.close(fig)

## Watch the best model

This is a deterministic rollout of `best_model.zip` on `normal_1`, a final test
case that was excluded from checkpoint selection. The MP4 captures every physics
frame, including intermediate frames during action persistence and the terminal
frame. The recorder streams frames to disk rather than storing them all in memory.

In [ ]:
import imageio.v2 as imageio
from record_video import VideoFrames

demo_case = get_case("normal_1")
video_path = RUN / "best_model_demo.mp4"
VIDEO_FPS = 50
with imageio.get_writer(str(video_path), fps=VIDEO_FPS, codec="libx264", macro_block_size=1) as writer:
    case_kwargs = dict(ENV_KWARGS)
    case_kwargs.update(demo_case["env_kwargs"])
    case_kwargs["continuous"] = CONTINUOUS
    base = CarRacingObstacles(max_episode_steps=demo_case["max_steps"], **case_kwargs)
    recorded = ActionRepeat(VideoFrames(base, writer), ACTION_REPEAT)
    video_env = DummyVecEnv([lambda: recorded])
    if HISTORY > 1:
        video_env = VecFrameStack(video_env, n_stack=HISTORY, channels_order="last")
    video_env = VecTransposeImage(video_env)
    try:
        best = PPO.load(str(RUN / "best_model.zip"), env=video_env, device=DEVICE)
        video_result = episode(best, video_env, demo_case["seed"])
    finally:
        video_env.close()
(RUN / "video_result.json").write_text(json.dumps(dict(case=demo_case["id"], **video_result), indent=2))
plt.close("all")
print(video_result)
display(Video(str(video_path), embed=True, width=640))

## Final evaluation: five + five fixed cases

Evaluate the selected checkpoint once on all ten supplied cases. Keep failed
episodes. The CSV and JSON contain every return and each group's mean, standard
deviation and success rate. These scores do not select another checkpoint.

In [ ]:
best = PPO.load(str(RUN / "best_model.zip"), device=DEVICE)
results = []
for case in load_cases():
    case_kwargs = dict(ENV_KWARGS)
    case_kwargs.update(case["env_kwargs"])
    case_kwargs["continuous"] = CONTINUOUS
    base = CarRacingObstacles(max_episode_steps=case["max_steps"], **case_kwargs)
    repeated = ActionRepeat(base, ACTION_REPEAT)
    test_env = DummyVecEnv([lambda: repeated])
    if HISTORY > 1:
        test_env = VecFrameStack(test_env, n_stack=HISTORY, channels_order="last")
    test_env = VecTransposeImage(test_env)
    try:
        result = episode(best, test_env, case["seed"])
        results.append(dict(case_id=case["id"], group=case["group"], **result))
    finally:
        test_env.close()
summary = save_results(results, RUN / "best_model_evaluation")

## Download the results (optional)

The archive includes the final and best models, model-selection metadata,
configuration, training/validation logs, reward graph, test statistics and video.

In [ ]:
def download_run():
    import shutil
    from google.colab import files
    archive = shutil.make_archive(str(RUN), "zip", root_dir=RUN)
    files.download(archive)

# Uncomment to download all run artifacts:
# download_run()